In [8]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from rich import print as rprint

# Tool Usage

## Invoke directly

In [3]:
@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息
    参数:
    city: 城市名称，如"北京"、"上海"
    返回:
    天气信息字符串
    """
    # 你的实现
    return city + "晴天，温度 15°C"

In [4]:
tool_invoke_result = get_weather.invoke({"city": "Shanghai"})
print(tool_invoke_result)
print(type(tool_invoke_result))

Shanghai晴天，温度 15°C
<class 'str'>


## 绑定到模型（主流）

In [9]:
load_dotenv(override=True)
model = init_chat_model(model="deepseek:deepseek-v4-flash")
model_with_tools = model.bind_tools([get_weather])

messages = [HumanMessage(content="What's weather at Shanghai?")]
ai_message = model_with_tools.invoke(messages)
rprint(ai_message)
print(ai_message.tool_calls)

AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': '用户想知道上海的天气。我可以使用get_weather工具来获取。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 59,
            'prompt_tokens': 299,
            'total_tokens': 358,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 14,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
            'prompt_cache_hit_tokens': 256,
            'prompt_cache_miss_tokens': 43
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
        'id': 'c8493495-0dd1-4d53-8d7f-248f63da5767',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--019f841a-5d5e-7f11-b967-e28f7425643e-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '上海'},
            'id': 'call_00_2kOOL1SIiHKiYgke5FY86785',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 299,
        'output_tokens': 59,
        'total_tokens': 358,
        'input_token_details': {'cache_read': 256},
        'output_token_details': {'reasoning': 14}
    }
)

[{'name': 'get_weather', 'args': {'city': '上海'}, 'id': 'call_00_2kOOL1SIiHKiYgke5FY86785', 'type': 'tool_call'}]


## 从Message流转看工具的调用

In [12]:
messages = [HumanMessage(content="What's weather at Shanghai?")]
ai_message = model_with_tools.invoke(messages)
# 添加AIMessage
messages.append(ai_message)
tool_calls = ai_message.tool_calls
for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 返回的是ToolMessage类型消息
        tool_response = get_weather.invoke(tool_call)
print(type(tool_response))
messages.append(tool_response)

print("=====================> messages <=====================")
for msg in messages:
    msg.pretty_print()
    rprint(msg)
print("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

<class 'langchain_core.messages.tool.ToolMessage'>
=====================> messages <=====================
================================ Human Message =================================

What's weather at Shanghai?


HumanMessage(content="What's weather at Shanghai?", additional_kwargs={}, response_metadata={})

================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_pPvTZxo8A6XzusTOY4XW5667)
 Call ID: call_00_pPvTZxo8A6XzusTOY4XW5667
  Args:
    city: 上海


AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': '用户想知道上海的天气。我需要使用get_weather工具来获取天气信息。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 61,
            'prompt_tokens': 299,
            'total_tokens': 360,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 16,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
            'prompt_cache_hit_tokens': 256,
            'prompt_cache_miss_tokens': 43
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
        'id': '4821ef36-e03b-4c15-91d8-5591f7ae2e71',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--019f841e-6485-7910-a4f6-bbb39e50c16e-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '上海'},
            'id': 'call_00_pPvTZxo8A6XzusTOY4XW5667',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 299,
        'output_tokens': 61,
        'total_tokens': 360,
        'input_token_details': {'cache_read': 256},
        'output_token_details': {'reasoning': 16}
    }
)

================================= Tool Message =================================
Name: get_weather

上海晴天，温度 15°C


ToolMessage(content='上海晴天，温度 15°C', name='get_weather', tool_call_id='call_00_pPvTZxo8A6XzusTOY4XW5667')

=====================> messages <=====================
final_response: 
content='上海目前是 **晴天**，气温 **15°C**。天气不错，适合外出活动！' additional_kwargs={'refusal': None, 'reasoning_content': '上海的天气是晴天，温度15°C。'} response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 380, 'total_tokens': 412, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 10, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 124}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'd1876032-d002-45ab-8497-13da1082d527', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f841e-6a0d-71e2-8444-1950105446a3-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 380, 'output_tokens': 32, 'total_tokens': 412, 'input_token_details': {